# Lab 2: Kiểm Định ADF & Chuẩn Bị Mô Hình ARIMA / SARIMA
---
## Bài 1: Khám phá dữ liệu (EDA)
- Load dữ liệu
- Chuyển date thành index thời gian
- Vẽ đồ thị chuỗi thời gian

**Câu hỏi cần trả lời:**
- Chuỗi có xu hướng không?
- Có dấu hiệu mùa vụ không?
- Chuỗi có khả năng dừng không?

In [ ]:
# [Cell 1]: Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Tắt cảnh báo FutureWarning từ statsmodels
warnings.filterwarnings('ignore', category=FutureWarning)

from statsmodels.tsa.stattools import adfuller

# Cấu hình giao diện biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

### 1. Load dữ liệu (Riêng)

In [ ]:
# [Cell 2]: Load dữ liệu từ thư mục raw
data_path = '../data/raw/a10.csv'  # Hoặc 'data/raw/a10.csv' tùy thư mục làm việc

df = pd.read_csv(data_path)

print("=== 5 DÒNG ĐẦU TIÊN CỦA DỮ LIỆU ===")
display(df.head())

print("\n=== THÔNG TIN DỮ LIỆU BAN ĐẦU ===")
df.info()

print("\n=== THỐNG KÊ MÔ TẢ ===")
display(df.describe())

### 2. Chuyển date thành index thời gian (Riêng)

In [ ]:
# [Cell 3]: Chuyển cột date thành DatetimeIndex
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df = df.asfreq('MS')  # Tần số theo tháng (Month Start)

print("Kiểu index:", type(df.index))
print("Tần số (Frequency):", df.index.freq)
print("Khoảng thời gian:", df.index.min().strftime('%Y-%m'), "đến", df.index.max().strftime('%Y-%m'))
display(df.head())

# Lưu dữ liệu sau khi set index vào data/processed/
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/a10_processed.csv', index=True)
print("\n-> Đã lưu dữ liệu vào: data/processed/a10_processed.csv")

### 3. Vẽ đồ thị chuỗi thời gian (Riêng)

In [ ]:
# [Cell 4]: Vẽ đồ thị chuỗi thời gian
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['value'], color='#1f77b4', linewidth=2, label='Doanh số bán thuốc hàng tháng (a10)')

plt.title('Đồ Thị Chuỗi Thời Gian Doanh Số Bán Hàng (Monthly Sales Data - a10)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Thời gian (Năm - Tháng)', fontsize=12, labelpad=10)
plt.ylabel('Doanh số (Triệu USD)', fontsize=12, labelpad=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

### Câu hỏi cần trả lời (Bài 1):

1. **Chuỗi có xu hướng không?**
   - **Có**: Chuỗi có **xu hướng tăng rõ rệt (Upward Trend)** theo thời gian từ năm 1991 đến 2008.

2. **Có dấu hiệu mùa vụ không?**
   - **Có**: Xuất hiện tính chu kỳ/mùa vụ lặp lại đều đặn hàng năm ($s = 12$ tháng), đỉnh thường rơi vào tháng 12 - tháng 1 và đáy rơi vào tháng 2. Biên độ dao động mùa vụ tăng dần theo thời gian.

3. **Chuỗi có khả năng dừng không?**
   - **Không**: Chuỗi **không có tính dừng (Non-stationary)** vì trung bình (mean) tăng theo thời gian và phương sai (variance) thay đổi / biên độ dao động mùa vụ mở rộng dần theo thời gian.

---
## Bài 2: Kiểm định ADF trên chuỗi gốc
- Thực hiện kiểm định ADF cho chuỗi gốc
  - Giá trị ADF statistic
  - p-value
  - Kết luận: Dừng / Không dừng

**Câu hỏi cần trả lời:**
- $H_0$ của ADF là gì?
- p-value cho biết điều gì?
- Chuỗi có dừng không?

In [ ]:
# [Cell 5]: Định nghĩa hàm kiểm định ADF (tái sử dụng xuyên suốt)
def adf_test(series, name='Chuỗi'):
    """
    Thực hiện kiểm định Augmented Dickey-Fuller (ADF) và in kết quả chi tiết
    """
    series_clean = series.dropna()
    result = adfuller(series_clean, autolag='AIC')
    
    adf_stat = result[0]
    p_value = result[1]
    used_lag = result[2]
    n_obs = result[3]
    critical_values = result[4]
    
    print(f"==================================================")
    print(f"  KẾT QUẢ KIỂM ĐỊNH ADF CHO: {name.upper()}")
    print(f"==================================================")
    print(f"• Test Statistic (ADF) : {adf_stat:.6f}")
    print(f"• p-value              : {p_value:.6f}")
    print(f"• Số độ trễ đã dùng    : {used_lag}")
    print(f"• Số lượng quan sát    : {n_obs}")
    print(f"• Các giá trị tới hạn (Critical Values):")
    for level, value in critical_values.items():
        print(f"    - Mức {level:>3}: {value:.6f}")
    print(f"--------------------------------------------------")
    
    alpha = 0.05
    if p_value <= alpha:
        print(f" Kết luận: p-value <= {alpha} => BÁC BỎ H0.")
        print(f"==> Chuỗi '{name}' LÀ CHUỖI DỪNG (Stationary).")
        is_stationary = True
    else:
        print(f" Kết luận: p-value > {alpha} => KHÔNG THỂ BÁC BỎ H0.")
        print(f"==> Chuỗi '{name}' KHÔNG DỪNG (Non-Stationary).")
        is_stationary = False
    print(f"==================================================\n")
    
    return {
        'adf_stat': adf_stat,
        'p_value': p_value,
        'critical_values': critical_values,
        'is_stationary': is_stationary
    }

In [ ]:
# [Cell 6]: Thực hiện kiểm định ADF trên chuỗi gốc (df['value'])
adf_original = adf_test(df['value'], name='Chuỗi gốc a10 (Doanh số bán thuốc)')

### Câu hỏi cần trả lời (Bài 2):

1. **$H_0$ của ADF là gì?**
   - Giả thuyết không ($H_0$): Chuỗi có **nghiệm đơn vị (Unit Root)** $\rightarrow$ Chuỗi **KHÔNG DỪNG (Non-Stationary)**.
   - Giả thuyết đối ($H_1$): Chuỗi không có nghiệm đơn vị $\rightarrow$ Chuỗi **CÓ TÍNH DỪNG (Stationary)**.

2. **p-value cho biết điều gì?**
   - p-value phản ánh xác suất ủng hộ giả thuyết $H_0$. Với mức ý nghĩa chuẩn $\alpha = 0.05$:
     - Nếu **p-value $\le 0.05$**: Bác bỏ $H_0$ $\rightarrow$ Chuỗi DỪNG.
     - Nếu **p-value $> 0.05$**: Không thể bác bỏ $H_0$ $\rightarrow$ Chuỗi KHÔNG DỪNG.

3. **Chuỗi có dừng không?**
   - **Kết luận: Chuỗi gốc KHÔNG DỪNG (Non-Stationary)**.
   - Vì p-value $\approx 1.000000 > 0.05$ và giá trị Test Statistic ($+3.145$) lớn hơn nhiều so với các giá trị tới hạn Critical Values (ở mức 5% là $-2.88$), không đủ bằng chứng bác bỏ $H_0$.

---
## Bài 3: Sai phân & kiểm định lại
- Sai phân bậc 1
- Vẽ lại đồ thị chuỗi sau sai phân
- Kiểm định ADF lần 2

**Câu hỏi cần trả lời:**
- Sau sai phân, chuỗi đã dừng chưa?
- Giá trị d trong ARIMA là bao nhiêu?

In [ ]:
# [Cell 7]: Sai phân bậc 1 (First-order Differencing)
df['value_diff1'] = df['value'].diff()

print("=== 5 DÒNG ĐẦU TIÊN SAU KHI SAI PHÂN BẬC 1 ===")
display(df[['value', 'value_diff1']].head())

print("\n=== SỐ LƯỢNG GIÁ TRỊ NaN ===")
print("Số giá trị NaN:", df['value_diff1'].isna().sum())

In [ ]:
# [Cell 8]: Vẽ lại đồ thị chuỗi sau sai phân bậc 1
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['value_diff1'], color='#2ca02c', linewidth=1.5, label='Chuỗi sau sai phân bậc 1 (d=1)')

plt.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='Mức trung bình cố định (Mean = 0)')

plt.title('Đồ Thị Chuỗi Thời Gian Sau Khi Sai Phân Bậc 1 (First-order Differencing)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Thời gian (Năm - Tháng)', fontsize=12, labelpad=10)
plt.ylabel('Giá trị sai phân', fontsize=12, labelpad=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# [Cell 9]: Kiểm định ADF lần 2 (tận dụng lại hàm adf_test từ Bài 2)
adf_diff1 = adf_test(df['value_diff1'], name='Chuỗi sau sai phân bậc 1 (d=1)')

### Câu hỏi cần trả lời (Bài 3):

1. **Sau sai phân, chuỗi đã dừng chưa?**
   - **Về trực quan:** Xu hướng tăng đã bị triệt tiêu hoàn toàn, chuỗi dao động quanh mức trung bình 0 (`Mean ≈ 0`).
   - **Về kiểm định ADF:** Giá trị p-value giảm mạnh từ 1.0 xuống xấp xỉ 0.11 - 0.12. Chuỗi đã **dừng về mặt xu hướng (Trend Stationary)**, tuy nhiên vẫn còn yếu tố mùa vụ lặp lại đều đặn hàng năm.

2. **Giá trị d trong ARIMA là bao nhiêu?**
   - **Trả lời: $d = 1$.**
   - Vì chúng ta áp dụng 1 lần sai phân bậc 1 ($Y_t' = Y_t - Y_{t-1}$) để đưa chuỗi về mức trung bình ổn định.